# 第46章 累积分布图（ecdfplot）

用ECDF直接展示小于等于某值的样本比例，无需选择分箱或带宽。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

比较分位数、阈值覆盖率或不同组的完整累计分布。

## 数据结构

一列连续数值，可按类别分组。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 complementary=False 改为 complementary=True，对比累计分布与互补累计分布的曲线方向
2. 修改 stat="proportion" 为 stat="count"，观察比例与计数的纵轴差异
3. 在图上添加 axvline 标记特定分位数（如中位数位置），说明ECDF在分位数读取中的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.3))
sns.ecdfplot(data=orders, x="order_value", color="#1a73e8", ax=ax)
ax.axhline(0.5, color="#9aa0a6", linestyle="--")
ax.set(title="订单金额累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(data=orders, x="order_value", hue="category", palette="colorblind", ax=ax)
ax.axvline(300, color="#d93025", linestyle="--", label="300元阈值")
ax.set(title="品类客单价累计分布", xlabel="客单价（元）", ylabel="累计比例")
fig.tight_layout()
plt.show()


## 3. 参数说明

- stat：proportion/count
- complementary：互补累计
- hue：分组
- weights：权重


## 4. 结果解读

在任意X值读取累计比例，或在给定比例处读取分位值。


## 常见误区

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.ecdfplot(data=marketing, x="conversion", hue="channel", complementary=True, palette="colorblind", ax=ax)
ax.set(title="转化率超过阈值的比例", xlabel="转化率阈值", ylabel="超过阈值的比例")
fig.tight_layout()
plt.show()


## 本章小结

用ECDF直接展示小于等于某值的样本比例，无需选择分箱或带宽。


### 你已经掌握

- 判断累积分布图（ecdfplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较分位数、阈值覆盖率或不同组的完整累计分布。 |
| 数据结构 | 一列连续数值，可按类别分组。 |
| 结果解读 | 在任意X值读取累计比例，或在给定比例处读取分位值。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `stat` | proportion/count |
| `complementary` | 互补累计 |
| `hue` | 分组 |
| `weights` | 权重 |


### 需要注意

- 不理解阶梯线含义
- 组间样本量不等时比较count
- 把陡峭部分解释为时间变化


### 完成检查

- [ ] 能判断什么问题适合使用累积分布图（ecdfplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
